# IALS (Implicit Alternating Least Squares )

В этом ноутбуке я построю IALS модель, протестирую её и сохраню 

## Импорт библиотек 

In [77]:
import pandas as pd
import numpy as np
import scipy 
import os
import implicit 
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from implicit.evaluation import train_test_split as implicit_split 
from implicit.evaluation import precision_at_k, mean_average_precision_at_k, ranking_metrics_at_k
import implicit.gpu


In [78]:
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

## Загрузка данных

#### Выгружаем матрицу

In [79]:
results_path = Path("../../../results/matrices")
user_item_sparse = scipy.sparse.load_npz(results_path/'artnail_user_item_sparse.npz')

#### Выгружаем очищенные таблицы

In [80]:
clean_path = Path("../../../Tables/CleanTable")
users_clean = pd.read_csv(clean_path / 'users_clean_final.csv')
items_clean = pd.read_csv(clean_path / 'items_clean_final.csv')

In [81]:
f'Размеры матрицы: {user_item_sparse.shape}'

'Размеры матрицы: (2893, 258)'

## IALS model 

#### Train/Validation split для метрик

In [82]:
train_sparse, val_sparse = implicit_split(user_item_sparse, train_percentage=0.9, random_state=42)

In [83]:
print(f"  Train: {train_sparse.nnz} ({train_sparse.nnz/user_item_sparse.nnz:.1%})")
print(f"  Val:   {val_sparse.nnz} ({val_sparse.nnz/user_item_sparse.nnz:.1%})")

  Train: 877 (90.4%)
  Val:   93 (9.6%)


#### Построение базовой модели 

In [84]:
IALS_model = model = implicit.als.AlternatingLeastSquares(
    factors=64,           # Размер эмбеддингов (пользователи × услуги)
    regularization=0.1,   # L2 регуляризация (0.01-0.5)
    iterations=50,        # ALS итераций (сходимость)
    random_state=42,      # Репродуцируемость
    use_gpu=False,        # CPU 
    num_threads=4         # Потоки (4-8 оптимально)
)

Обучение 

In [85]:
CONFIDENCE_SCALE = 15
IALS_model.fit(train_sparse*CONFIDENCE_SCALE)

print(f"  User embeddings: {model.user_factors.shape}")
print(f"  Item embeddings: {model.item_factors.shape}")

  0%|          | 0/50 [00:00<?, ?it/s]

  User embeddings: (2893, 64)
  Item embeddings: (258, 64)


## Оценка качества базовой модели 

IALS будет выполнять функции подбора кандидатов, а функцию ранжирования будет выполнять CutBoost модель, поэтому в оценке IALS главной метрикой буду считать Recall@K, где K будет большим.

Цель не отобрать конкретные 10 лучших вариантов, а отобрать 50 кандидатов, главное чтобы среди них были хорошие кандидаты и CutBoost увидел их

Проверка типов 

In [86]:
print(type(val_sparse.T.tocsr().indices[0]),
type(train_sparse.T.tocsr().indices[0]),
type(train_sparse.T.tocsr().indptr[0]),
type(val_sparse.T.tocsr().indptr[0]))

<class 'numpy.int32'> <class 'numpy.int32'> <class 'numpy.int32'> <class 'numpy.int32'>


In [87]:
p50 = precision_at_k(IALS_model, val_sparse, train_sparse, K=50, num_threads=4)
f'Precision@50 = {p50}'

  0%|          | 0/683 [00:00<?, ?it/s]

AttributeError: 'implicit.evaluation._memoryviewslice' object has no attribute 'dtype'